In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
cw_path = "/green-projects/project-urban_colocation_intelligence/workspace/share/data/locomizer_filtered_output/canarywharf_r10_2025-03_04.csv"
home_path = "/green-projects/project-urban_colocation_intelligence/workspace/share/data/locomizer_filtered_output/visitor_home_locations.csv"

cols = ["id", "ts", "type"]

df = pd.read_csv(
    cw_path,
    usecols=cols,
    parse_dates=["ts"]
)

home_df = pd.read_csv(home_path)

/tmp/ipykernel_2281325/2069086050.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(


In [4]:
df = df[df["type"] == "stop"].copy()

print("After stop filter:", len(df))

After stop filter: 1876684


In [19]:
df['id'].nunique()

157101

In [5]:
df["ts"] = pd.to_datetime(df["ts"], unit="s")
df["hour"] = df["ts"].dt.hour
df["weekday"] = df["ts"].dt.weekday
df["date"] = df["ts"].dt.date

/tmp/ipykernel_2281325/2720803557.py:1: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df["ts"] = pd.to_datetime(df["ts"], unit="s")


In [7]:
workers = df[
    (df["weekday"] < 5) &
    (df["hour"].between(8, 18))
]

In [8]:
active_days = workers.groupby("id")["date"].nunique()

In [9]:
repetitive_ids = active_days[active_days >= 3].index

In [10]:
visit_counts = workers.groupby("id").size()

In [11]:
likely_workers_ids = visit_counts[
    (visit_counts >= 5) &
    (visit_counts.index.isin(repetitive_ids))
].index

print("Likely workers:", len(likely_workers_ids))

Likely workers: 9962


In [12]:
likely_workers_list = likely_workers_ids.tolist()

In [13]:
print(likely_workers_list[:10])

['0004149d28cf68bafa4235d888bf148f', '000c29b9516def0bec8cf107234ee1f1', '00145899d6aaa6a4a203a5bb332ddbfb', '001e5b171a43a454fbe96d4d826a9e04', '0022aaa1e46e3b4bae313f35af842a92', '002fd365a60735e2fbb693af2a5a1db6', '00325c05342d6e81beb8f71f4081f258', '003548414e212615c959a27415a19891', '0036e90f1e17c9f898f77182b5031751', '003afdaf307e38d05f5bf0180b5ed7e8']


In [14]:
likely_workers_df = pd.DataFrame({"user_id": likely_workers_ids})

In [15]:
likely_workers_df.to_csv("likely_workers_ids.csv", index=False)

In [16]:
worker_home = home_df[
    home_df["id"].isin(likely_workers_ids)
].copy()

print("Workers with home:", worker_home["id"].nunique())

Workers with home: 9962


In [18]:
# WITHOUT active days filter (relaxed workers)

likely_workers_ids_relaxed = visit_counts[
    visit_counts >= 5
].index

print("Likely workers (no active day filter):", len(likely_workers_ids_relaxed))

Likely workers (no active day filter): 33178
